In [13]:
from src import api
from my_notebook.apiclient import APIClient
from my_notebook.structs import ExperimentInput, ExperimentSetting, ExperimentResult, ExperimentRecord, ExperimentPrompt, ExperimentMetadata, ExperimentLanguage, ExperimentRunMetadataV1
from datetime import datetime, UTC
from dataclasses import asdict
from pathlib import Path
from time import perf_counter
import sys

In [14]:
def run_experiment(
    setting: ExperimentSetting, input: ExperimentInput, allow_fail: bool = False
) -> ExperimentRecord:
    start_time = perf_counter()
    try:

        translation = client.translate(
            text=input.text,
            source_language=input.language.source_language,
            target_language=input.language.target_language,
            model=setting.model,
            prompt_setting=setting.prompt.setting,
        )

        rating = client.rate(source=input.text, translation=translation)
    except Exception as e:
        if allow_fail:
            end_time = perf_counter()

            metadata = ExperimentMetadata(
                created_at=datetime.now(UTC).isoformat(),
                success=False,
                error=str(e),
                elapsed_seconds=end_time - start_time,
            )
            print(f"Experiment failed: {e}", file=sys.stderr)
            return ExperimentRecord(
                result=None, input=input, setting=setting, metadata=metadata
            )
        else:
            raise

    end_time = perf_counter()

    result = ExperimentResult(translation=translation, rating=rating)

    metadata = ExperimentMetadata(
        created_at=datetime.now(UTC).isoformat(),
        success=True,
        error=None,
        elapsed_seconds=end_time - start_time,
    )

    record = ExperimentRecord(
        result=result, input=input, setting=setting, metadata=metadata
    )

    return record

In [ ]:
with open(".runpod-fastapi-token") as file:
    token = file.read().strip()
# client = APIClient(setting=APIClient.Setting(token="default_token"))
client = APIClient(setting=APIClient.Setting(token=token, base_url="https://hju3d8fhhukees-8000.proxy.runpod.net/"))

In [16]:
models = client.models()
models

HTTP 404: 


Exception: 

In [ ]:
def text_loader(path: str):
    with open(path) as file:
        return file.read()

prompts = [
    ExperimentPrompt(
        name="minimal",
        setting=api.PromptSetting(
            system_prompt=(
                "You are an assistant who translates. "
                "The first line of the input is source and target language specification."
            ),
            prompt_format="{source_language} -> {target_language}\n{text}",
        ),
    ),
    ExperimentPrompt(
        name="professional",
        setting=api.PromptSetting(
            system_prompt="""You are a professional translation engine.

Translate the user's text faithfully from the specified source language to the specified target language.

Rules:
- Preserve the meaning exactly.
- Do not answer the user's request.
- Do not summarize.
- Do not explain.
- Do not add or remove information.
- Preserve formatting where possible.
- Return only the translation text.""",
            prompt_format="""Source language: {source_language}
Target language: {target_language}

Text:
{text}""",
        ),
    ),
    ExperimentPrompt(
        name="verbose",
        setting=api.PromptSetting(
            system_prompt=text_loader("my_notebook/verbose.txt"),
            prompt_format="""Translate the following text.

Source language: {source_language}
Target language: {target_language}

Input:
{text}

Translation:""",
        ),
    ),
]

In [ ]:
import json
texts = []
for i in range(78, 115):
    data = json.load(open("../../public/quran/exegesis/aliquli/en-US/{}.json".format(i)))
    texts.extend(data["translations"].values())
texts[:10]

['What is it about which they question each other?!',
 '[Is it] about the great tiding,',
 'the one about which they differ?',
 'No indeed! They will soon know!',
 'Again, no indeed! They will soon know!',
 'Did We not make the earth a resting place?',
 'and the mountains stakes?',
 'and create you in pairs?<{["F", 1]}>',
 'and make your sleep for rest?',
 'and make the night a covering?']

In [ ]:
language = ExperimentLanguage(source_language="english", target_language="indonesian")
metadata = ExperimentRunMetadataV1(
    created_at=datetime.now(UTC).isoformat(),
    source_language=language.source_language,
    target_language=language.target_language,
    models=models,
    prompt_names=[prompt.name for prompt in prompts],
    prompt_settings={prompt.name: prompt.setting for prompt in prompts},
    sample_count=len(texts),
    notes="TL of mirali exegesis with cloud model",
)


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base_path = Path("results")
base_path.mkdir(exist_ok=True)
record_filename = base_path / f"{timestamp}.records.jsonl"
metadata_filename = base_path / f"{timestamp}.metadata.json"


with open(metadata_filename, "w", encoding="utf-8") as f:
    json.dump(asdict(metadata), f, ensure_ascii=False, indent=2)


def append_record(path: str, record: ExperimentRecord):
    with open(path, "a", encoding="utf-8") as f:
        json.dump(asdict(record), f, ensure_ascii=False)
        f.write("\n")

In [ ]:
from contextlib import redirect_stderr

error_file_path = base_path / "errors.log"
error_file = open(error_file_path, "w", encoding="utf-8")

def count_errors(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)
    
with redirect_stderr(error_file):
    for ti, text in enumerate(texts, 1):
        input = ExperimentInput(text=text, language=language)
        for mi, model in enumerate(models, 1):
            for pi, prompt in enumerate(prompts, 1):
                setting = ExperimentSetting(model=model, prompt=prompt)

                print(
                    f"""\rText {ti}/{len(texts)} | Model {mi}/{len(models)} | Prompt {pi}/{len(prompts)} | Error line count: {count_errors(error_file_path)}""",
                    end="",
                    flush=True,
                )

                record = run_experiment(setting=setting, input=input, allow_fail=True)
                append_record(str(record_filename), record)

error_file.close()

Text 2/564 | Model 1/1 | Prompt 3/3 | Error line count: 0

KeyboardInterrupt: 